# Cooperative Coevolution for Interpretable Fraud Detection Rules
## TransUnion Capstone – IDS 560

---

### Motivation

The previous notebook evolved a **single AND-rule** — one root-to-leaf path in a decision tree.  
While effective, a single path limits coverage: fraud patterns are diverse and rarely captured by one rule.

This notebook evolves a **rule set**: up to 16 paths combined by OR logic.

```
RULE SET =
  (V14 <= -2.1 AND V10 <= -1.3)          ← path 1
  OR
  (V4 > 3.2 AND V12 <= -0.8 AND V9 <= -1.0)  ← path 2
  OR
  ...                                      ← up to 16 paths
```

Each path is a conjunction (AND) of up to 10 conditions — analogous to one root-to-leaf path in a decision tree.

---

### Why Coevolution?

A naive approach would be to evolve the full rule set as one monolithic individual.  
The problem: the search space explodes — 16 paths × 10 conditions = 160 variables to tune simultaneously.

**Cooperative Coevolution** splits this into two interacting populations:

| Population | What evolves | Fitness signal |
|---|---|---|
| **Path pool** | Individual paths (≤ 10 nodes) | Standalone F1 as a single-path rule |
| **Rule set pool** | Combinations of paths (≤ 16) | Combined F1 using OR logic |

The two populations evolve in alternating cycles and exchange individuals via **migration**:

```
Path Pool  ──[seed best paths]──►  Rule Set Pool
           ◄──[extract paths]────
```

- **Paths → Rule Sets**: proven standalone paths become building blocks in new rule sets  
- **Rule Sets → Paths**: paths that work well *in combination* get promoted even if weak alone

This lets the algorithm discover both **the best individual paths** and **the best combinations**.

## Step 1: Imports and Setup

In [56]:
import numpy as np
import pandas as pd
import random
import copy
import json
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    mutual_info_score, roc_auc_score, average_precision_score
)
from deap import base, creator, tools

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Step 2: Load Data

Same credit card fraud dataset as before.
- 284,807 transactions, ~0.17% fraud
- `Class = 1` → fraud, `Class = 0` → legitimate
- Features V1–V28 are PCA-transformed anonymized values

V10 binning happens **after** the train/val/test split (Step 3) so that bin boundaries
are fitted on training data only, with no leakage from validation or test rows.

In [57]:
import os

# ── Data path ─────────────────────────────────────────────────────────────────
# Update DATA_FILE if your CSV lives elsewhere.
DATA_FILE = os.path.join(os.path.dirname(os.path.abspath("__file__")), "creditcard.csv")

df = pd.read_csv(DATA_FILE)
df = df.drop_duplicates().reset_index(drop=True)
df["Class"] = df["Class"].astype(int)

print(f"Shape: {df.shape}")
print(f"Overall fraud rate: {df['Class'].mean():.4f}")

Shape: (283726, 31)
Overall fraud rate: 0.0017


## Step 3: Train / Validation / Test Split

- **Train (70%)** — used for feature engineering, screening, and threshold generation
- **Validation (15%)** — the fitness signal during evolution (both populations)
- **Test (15%)** — held out entirely; only touched for final reporting

The split is performed on **raw V10** (continuous). V10 binning is applied immediately
after, fitted on the training split only, preventing any leakage into val/test.

In [58]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_SEED
)

print("Train fraud rate:", y_train.mean())
print("Val   fraud rate:", y_val.mean())
print("Test  fraud rate:", y_test.mean())

Train fraud rate: 0.0016665995327479256
Val   fraud rate: 0.0016682722808336662
Test  fraud rate: 0.0016682722808336662


## Step 3b: V10 Binning (train-fit, applied to all splits)

V10 has the strongest non-linear fraud signal of any feature — its lowest decile has a
fraud rate ~60× the baseline. We replace it with an ordinal integer `V10_bin_code` (0–9).

**Why after the split?**
Fitting `pd.qcut` on the full dataset would use quantile boundaries computed from val and
test rows — a form of data leakage. The correct procedure is:
1. Fit decile boundaries on `X_train["V10"]` only
2. Apply those fixed edges to `X_val` and `X_test` via `pd.cut`

Out-of-range values (if val/test V10 falls outside training min/max) are handled by
extending the outermost edges to ±∞, so no NaNs are introduced.

In [59]:
# ── Fit bin edges on training V10 only ───────────────────────────────────────
_, bin_edges = pd.qcut(X_train["V10"], q=10, duplicates="drop", retbins=True)

# Extend outermost edges to ±∞ so val/test values outside the training range
# are still assigned a valid bin rather than producing NaN.
bin_edges[0]  = -np.inf
bin_edges[-1] =  np.inf

def apply_v10_binning(X, edges):
    """
    Replace the V10 column with V10_bin_code (integer 0–9) in the same column
    position, using pre-fitted bin edges from the training set.
    """
    codes    = pd.cut(X["V10"], bins=edges, labels=False, include_lowest=True).astype(int)
    idx      = list(X.columns).index("V10")
    X        = X.drop(columns=["V10"]).copy()
    X.insert(idx, "V10_bin_code", codes.values)
    return X

X_train = apply_v10_binning(X_train, bin_edges)
X_val   = apply_v10_binning(X_val,   bin_edges)
X_test  = apply_v10_binning(X_test,  bin_edges)

# ── Verify fraud rate by bin (training set) ───────────────────────────────────
v10_rates = X_train.assign(Class=y_train.values).groupby("V10_bin_code")["Class"].agg(["count", "mean"])
v10_rates.columns = ["n_transactions", "fraud_rate"]
print("Fraud rate by V10 decile bin (training set):")
print(v10_rates.to_string())
print(f"\nV10_bin_code range in val:  {X_val['V10_bin_code'].min()}–{X_val['V10_bin_code'].max()}")
print(f"V10_bin_code range in test: {X_test['V10_bin_code'].min()}–{X_test['V10_bin_code'].max()}")

Fraud rate by V10 decile bin (training set):
              n_transactions  fraud_rate
V10_bin_code                            
0                      19861    0.014400
1                      19861    0.000453
2                      19861    0.000151
3                      19860    0.000201
4                      19861    0.000101
5                      19861    0.000201
6                      19860    0.000252
7                      19861    0.000352
8                      19861    0.000201
9                      19861    0.000352

V10_bin_code range in val:  0–9
V10_bin_code range in test: 0–9


## Step 4: Feature Screening

With a larger rule structure (up to 16 × 10 = 160 conditions), controlling feature space is even more important.

We rank features by **Mutual Information** with the fraud label and keep the top K.  
Only the training set is used here — no leakage.

In [60]:
def feature_screen(Xtr, ytr, max_bins=10):
    rows = []
    for col in Xtr.columns:
        uniq = Xtr[col].nunique()
        if uniq < 3:
            continue
        try:
            b = pd.qcut(Xtr[col], q=min(max_bins, uniq), duplicates="drop")
        except:
            continue
        tmp = pd.DataFrame({"bin": b, "Class": ytr}).dropna()
        if tmp["bin"].nunique() < 2:
            continue
        spread = float(tmp.groupby("bin", observed=True)["Class"].mean().pipe(lambda s: s.max() - s.min()))
        mi = float(mutual_info_score(tmp["bin"].cat.codes, tmp["Class"]))
        rows.append({"feature": col, "mi": mi, "spread": spread})
    return pd.DataFrame(rows).sort_values(["mi", "spread"], ascending=False)

rank_df = feature_screen(X_train, y_train)
TOP_K = 12
TOP_FEATURES = rank_df["feature"].head(TOP_K).tolist()

# Guarantee V10_bin_code is always retained — it is a deliberate engineered feature
# and CATEGORICAL_FEATURES depends on it being present in X_train.
if "V10_bin_code" not in TOP_FEATURES:
    TOP_FEATURES.append("V10_bin_code")
    print("Note: V10_bin_code added to TOP_FEATURES (not in top-K by MI but required).")

X_train = X_train[TOP_FEATURES]
X_val   = X_val[TOP_FEATURES]
X_test  = X_test[TOP_FEATURES]

print("Selected features:", TOP_FEATURES)

Selected features: ['V14', 'V4', 'V10_bin_code', 'V12', 'V3', 'V11', 'V17', 'V16', 'V7', 'V2', 'V9', 'V27']


## Step 5: Evaluation Metrics

Helper functions for consistent reporting throughout the notebook.

In [61]:
def evaluate(y_true, y_pred):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }

def confusion(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return {
        "TP": int(((y_pred == 1) & (y_true == 1)).sum()),
        "FP": int(((y_pred == 1) & (y_true == 0)).sum()),
        "FN": int(((y_pred == 0) & (y_true == 1)).sum()),
        "TN": int(((y_pred == 0) & (y_true == 0)).sum()),
    }

## Step 6: Rule Representation

### Conditions, Paths, and Rule Sets

The GA works with three nested levels of structure:

```
CONDITION  →  (feature, operator, threshold)
               e.g. ("V14", "<=", -2.1)        ← continuous feature
               e.g. ("V10_bin_code", "=", 0)   ← categorical feature

PATH       →  [condition₁ AND condition₂ AND ... condition_k]   k ≤ 10
               One root-to-leaf walk in a decision tree
               Each feature appears at most once per path.

RULE SET   →  [path₁ OR path₂ OR ... OR path_n]                n ≤ 16
               The final business rule — a union of decision paths
```

### Operators

| Feature type | Allowed operators | Threshold domain |
|---|---|---|
| Continuous (V1–V28 except V10) | `>`, `>=`, `<`, `<=` | 19 training-set quantile points |
| Categorical (`V10_bin_code`) | `>`, `>=`, `=`, `<`, `<=` | Integers 0–9 (decile codes) |

### No-Duplicate Feature Constraint

Each feature appears **at most once per path**. Repeating the same feature wastes a
node slot — the second condition on the same feature is always subsumed by or redundant
with the first. Enforcing uniqueness keeps the 10-node budget fully productive.

### Greedy Rule Set Initialization

Rather than building all paths randomly, we initialize rule sets greedily:
each new path is chosen (from a small candidate pool) to maximize coverage of
fraud cases **not already caught** by the paths already in the rule set.
This seeds rule set diversity from the start — paths aim at different fraud subsets
rather than all converging on the same obvious region.
30% of rule sets are still initialized fully randomly to preserve population diversity.

In [62]:
# ── Structural constants ───────────────────────────────────────────────────────
MAX_PATHS = 16;  MIN_PATHS = 1
MAX_NODES = 10;  MIN_NODES = 1

FEATURES = list(X_train.columns)

CATEGORICAL_FEATURES = {"V10_bin_code"}
CONTINUOUS_OPS       = [">", ">=", "<", "<="]
CATEGORICAL_OPS      = [">", ">=", "=", "<", "<="]

OP_FUNCS = {
    ">":  lambda x, t: x > t,
    ">=": lambda x, t: x >= t,
    "=":  lambda x, t: x == t,
    "<":  lambda x, t: x < t,
    "<=": lambda x, t: x <= t,
}

# ── Threshold bank ─────────────────────────────────────────────────────────────
quantiles = np.linspace(0.05, 0.95, 19)
threshold_bank = {}
for col in FEATURES:
    if col in CATEGORICAL_FEATURES:
        threshold_bank[col] = np.arange(X_train[col].nunique(), dtype=float)
    else:
        threshold_bank[col] = np.quantile(X_train[col].values, quantiles)

# Pre-extract fraud rows from the training set once for greedy initialisation.
# Using only the ~350 fraud rows makes greedy init fast.
_fraud_mask_train = (np.asarray(y_train) == 1)
_fraud_X_train    = X_train[_fraud_mask_train].reset_index(drop=True)

# ── Primitive constructors ─────────────────────────────────────────────────────
def random_condition(exclude_features=None):
    """
    Draw one (feature, operator, threshold) triple.
    exclude_features: set of feature names already in the path — skipped to
    enforce the no-duplicate-feature constraint.
    """
    exclude = exclude_features or set()
    available = [f for f in FEATURES if f not in exclude]
    if not available:
        available = FEATURES          # fallback: all features (edge case)
    col     = random.choice(available)
    ops     = CATEGORICAL_OPS if col in CATEGORICAL_FEATURES else CONTINUOUS_OPS
    return (col, random.choice(ops), float(random.choice(threshold_bank[col])))

def random_path():
    """
    One root-to-leaf path: 1–MAX_NODES AND conditions.
    Each feature appears at most once (no-duplicate constraint).
    """
    k = random.randint(MIN_NODES, min(MAX_NODES, len(FEATURES)))
    cols = random.sample(FEATURES, k)          # sample without replacement
    return [
        (col,
         random.choice(CATEGORICAL_OPS if col in CATEGORICAL_FEATURES else CONTINUOUS_OPS),
         float(random.choice(threshold_bank[col])))
        for col in cols
    ]

def deduplicate_path(path):
    """
    Remove any later condition that reuses a feature already seen in the path.
    Called after crossover to restore the no-duplicate constraint.
    """
    seen, result = set(), []
    for cond in path:
        if cond[0] not in seen:
            seen.add(cond[0])
            result.append(cond)
    return result

def greedy_ruleset():
    """
    Build a rule set by greedily covering new fraud cases at each path slot.
    For each slot, generate 10 candidate paths and pick the one that covers
    the most fraud cases not already caught by previously selected paths.
    Evaluated only on the ~350 fraud training rows — fast and leakage-free.
    """
    n = random.randint(MIN_PATHS, MAX_PATHS)
    covered = np.zeros(len(_fraud_X_train), dtype=bool)
    ruleset = []

    for _ in range(n):
        candidates  = [random_path() for _ in range(10)]
        best_path, best_new = candidates[0], -1
        for path in candidates:
            fires     = apply_path(path, _fraud_X_train)
            new_count = int((fires & ~covered).sum())
            if new_count > best_new:
                best_new, best_path = new_count, path
        ruleset.append(best_path)
        covered |= apply_path(best_path, _fraud_X_train)

    return ruleset

def random_ruleset():
    """
    70% greedy coverage initialisation → diverse path coverage from the start.
    30% fully random → maintains population diversity.
    """
    if random.random() < 0.70:
        return greedy_ruleset()
    return [random_path() for _ in range(random.randint(MIN_PATHS, MAX_PATHS))]

# ── Inference logic ────────────────────────────────────────────────────────────
def apply_path(path, Xdf):
    """AND across all conditions; returns bool array (one entry per row)."""
    mask = np.ones(len(Xdf), dtype=bool)
    for col, op, thr in path:
        mask &= OP_FUNCS[op](Xdf[col].values, thr)
    return mask

def apply_ruleset(ruleset, Xdf):
    """OR across all paths; returns 0/1 integer array."""
    fired = np.zeros(len(Xdf), dtype=bool)
    for path in ruleset:
        fired |= apply_path(path, Xdf)
    return fired.astype(int)

# ── Pretty-print helpers ───────────────────────────────────────────────────────
def fmt_threshold(col, thr):
    return str(int(thr)) if col in CATEGORICAL_FEATURES else f"{thr:.4f}"

def pretty_path(path):
    return " AND ".join(f"{col} {op} {fmt_threshold(col, thr)}" for col, op, thr in path)

def pretty_ruleset(ruleset):
    parts = [f"  [{i+1:02d}] {pretty_path(p)}" for i, p in enumerate(ruleset)]
    return "\nOR\n".join(parts)

## Step 7: Fitness Functions

Each population has its own fitness signal.

### Path fitness — unchanged
Standalone F1 on the validation set, treating the path as a one-path rule set.

### Rule set fitness — three components

```
score = base_F1  +  λ × diversity_bonus  −  μ × parsimony_penalty
```

| Component | Weight | What it rewards |
|---|---|---|
| `base_F1` | 1.0 | Overall fraud detection on the validation set |
| `diversity_bonus` | λ = 0.05 | Paths that catch fraud cases no other path catches |
| `parsimony_penalty` | μ = 0.02 | Simpler rule sets when F1 is otherwise equal |

**Diversity bonus** — prevents all paths converging on the same obvious fraud subset.
For each path, count fraud cases it catches that *no other path* catches (unique TPs).
Normalise by (n_fraud_val × n_paths) so the bonus is always in [0, 1].

**Parsimony penalty** — mild preference for interpretable rule sets.
Average of: (n_paths / MAX_PATHS) and (avg_nodes_per_path / MAX_NODES).
At λ = 0.05 and μ = 0.02 these terms only break ties — a genuinely better F1 always wins.

In [63]:
# Weights for the multi-component rule set fitness
COVERAGE_LAMBDA = 0.05   # diversity bonus weight
PARSIMONY_MU    = 0.02   # parsimony penalty weight

# Pre-extract fraud indices from validation set once (reused every eval_ruleset call)
_fraud_idx_val = np.where(np.asarray(y_val) == 1)[0]

def eval_path(path):
    """
    Standalone fitness for a single path.
    Treats the path as a one-path rule set and measures validation F1.
    """
    y_pred = apply_path(path, X_val).astype(int)
    return (f1_score(y_val, y_pred, zero_division=0),)

def eval_ruleset(ruleset):
    """
    Multi-component fitness for a full rule set:
      score = base_F1  +  COVERAGE_LAMBDA × diversity_bonus
                       −  PARSIMONY_MU    × parsimony_penalty

    diversity_bonus:
        For each path, count fraud cases it catches that no other path catches.
        Normalised to [0, 1]. Penalises rule sets where all paths overlap.

    parsimony_penalty:
        Mean of (n_paths / MAX_PATHS) and (avg_nodes / MAX_NODES).
        Rewards simpler, more interpretable rule sets when F1 is tied.
    """
    # ── Base F1 ───────────────────────────────────────────────────────────────
    y_pred  = apply_ruleset(ruleset, X_val)
    base_f1 = f1_score(y_val, y_pred, zero_division=0)

    # ── Diversity bonus ───────────────────────────────────────────────────────
    if len(ruleset) > 1 and len(_fraud_idx_val) > 0:
        # fires[i] = bool array over fraud-only val rows for path i
        fires = [apply_path(p, X_val)[_fraud_idx_val] for p in ruleset]
        unique_total = 0
        for i, mask_i in enumerate(fires):
            others = np.zeros(len(_fraud_idx_val), dtype=bool)
            for j, mask_j in enumerate(fires):
                if j != i:
                    others |= mask_j
            unique_total += int((mask_i & ~others).sum())
        diversity_bonus = unique_total / (len(_fraud_idx_val) * len(ruleset))
    else:
        diversity_bonus = 0.0

    # ── Parsimony penalty ─────────────────────────────────────────────────────
    avg_nodes        = np.mean([len(p) for p in ruleset]) if ruleset else 0
    parsimony_penalty = (len(ruleset) / MAX_PATHS + avg_nodes / MAX_NODES) / 2

    score = base_f1 + COVERAGE_LAMBDA * diversity_bonus - PARSIMONY_MU * parsimony_penalty
    return (score,)

## Step 8: DEAP Setup — Two Populations

DEAP uses `creator` to define individual types and `Toolbox` to register operations.

We create **two independent toolboxes** — one per population — so each population  
has its own selection, crossover, and mutation operators tuned to its structure.

```
path_tb    →  operates on Path individuals  (list of conditions)
rs_tb      →  operates on Ruleset individuals (list of paths)
```

In [64]:
# Guard against re-registration if the cell is re-run
if "PathFitness" not in creator.__dict__:
    creator.create("PathFitness", base.Fitness, weights=(1.0,))
if "Path" not in creator.__dict__:
    creator.create("Path", list, fitness=creator.PathFitness)

if "RulesetFitness" not in creator.__dict__:
    creator.create("RulesetFitness", base.Fitness, weights=(1.0,))
if "Ruleset" not in creator.__dict__:
    creator.create("Ruleset", list, fitness=creator.RulesetFitness)

# ── Path toolbox ──────────────────────────────────────────────────────────────
path_tb = base.Toolbox()
path_tb.register("individual", tools.initIterate, creator.Path, random_path)
path_tb.register("population", tools.initRepeat, list, path_tb.individual)
path_tb.register("evaluate",   eval_path)
path_tb.register("select",     tools.selTournament, tournsize=3)
path_tb.register("clone",      copy.deepcopy)

# ── Rule set toolbox ──────────────────────────────────────────────────────────
rs_tb = base.Toolbox()
rs_tb.register("individual", tools.initIterate, creator.Ruleset, random_ruleset)
rs_tb.register("population", tools.initRepeat, list, rs_tb.individual)
rs_tb.register("evaluate",   eval_ruleset)
rs_tb.register("select",     tools.selTournament, tournsize=3)
rs_tb.register("clone",      copy.deepcopy)

print("Path toolbox and Ruleset toolbox ready.")

Path toolbox and Ruleset toolbox ready.


## Step 9: Genetic Operators

### Path-level operators

These operate on individual paths (lists of conditions).

**Mutation** (5 fine-grained operations):

| Op | What changes |
|---|---|
| `thr` | Replace the threshold with another from the bank — same feature, same operator |
| `flip` | Switch to a different operator from the feature's allowed pool (e.g. `<=` → `<`, or `=` → `>` for V10_bin_code) |
| `replace` | Swap the entire condition for a brand-new random one |
| `add_node` | Append a new condition (path becomes more specific) |
| `drop_node` | Remove one condition (path becomes more general) |

`flip` is operator-aware: continuous features cycle through `{>, >=, <, <=}`;
categorical features cycle through `{>, >=, =, <, <=}`.

**Crossover**: swap one condition between two paths (fine-grained recombination).

---

### Rule set-level operators

These operate on rule sets (lists of paths).

**Mutation** (3 coarse-grained operations):

| Op | What changes |
|---|---|
| `add_path` | Append a new random path (expands coverage) |
| `drop_path` | Remove one path (reduces noise / false positives) |
| `mutate_node` | Pick a random path inside the rule set, apply one path-level mutation |

**Crossover**: swap one entire path between two rule sets (coarse-grained recombination).

In [65]:
# ── Path operators ────────────────────────────────────────────────────────────
def mutate_path(path):
    op = random.choice(["thr", "flip", "replace", "add_node", "drop_node"])

    if op == "thr" and len(path) > 0:
        # Shift threshold; keep feature and operator unchanged
        i = random.randrange(len(path))
        col, cur_op, _ = path[i]
        path[i] = (col, cur_op, float(random.choice(threshold_bank[col])))

    elif op == "flip" and len(path) > 0:
        # Switch to a different operator from the feature's allowed pool
        i = random.randrange(len(path))
        col, cur_op, thr = path[i]
        ops_pool  = CATEGORICAL_OPS if col in CATEGORICAL_FEATURES else CONTINUOUS_OPS
        other_ops = [o for o in ops_pool if o != cur_op]
        path[i]   = (col, random.choice(other_ops), thr)

    elif op == "replace" and len(path) > 0:
        # Replace one condition; pick a feature not already in the path
        i        = random.randrange(len(path))
        used     = {c for c, _, _ in path} - {path[i][0]}   # exclude current slot's feature
        path[i]  = random_condition(exclude_features=used)

    elif op == "add_node" and len(path) < MAX_NODES:
        # Add a condition on a feature not yet in the path
        used = {c for c, _, _ in path}
        cond = random_condition(exclude_features=used)
        path.append(cond)

    elif op == "drop_node" and len(path) > MIN_NODES:
        path.pop(random.randrange(len(path)))

    return (path,)

def cx_paths(p1, p2):
    """
    Swap one condition between two paths.
    Deduplicate afterwards to restore the no-duplicate-feature constraint
    in case the swapped condition introduces a feature already present.
    """
    if len(p1) > 0 and len(p2) > 0:
        i, j = random.randrange(len(p1)), random.randrange(len(p2))
        p1[i], p2[j] = copy.deepcopy(p2[j]), copy.deepcopy(p1[i])
        p1[:] = deduplicate_path(p1)
        p2[:] = deduplicate_path(p2)
    return p1, p2

path_tb.register("mate",   cx_paths)
path_tb.register("mutate", mutate_path)

# ── Rule set operators ────────────────────────────────────────────────────────
def mutate_ruleset(ruleset):
    op = random.choice(["add_path", "drop_path", "mutate_node"])
    if   op == "add_path"    and len(ruleset) < MAX_PATHS:
        ruleset.append(random_path())
    elif op == "drop_path"   and len(ruleset) > MIN_PATHS:
        ruleset.pop(random.randrange(len(ruleset)))
    elif op == "mutate_node" and len(ruleset) > 0:
        mutate_path(ruleset[random.randrange(len(ruleset))])
    return (ruleset,)

def cx_rulesets(r1, r2):
    """Swap one entire path between two rule sets."""
    if len(r1) > 0 and len(r2) > 0:
        i, j = random.randrange(len(r1)), random.randrange(len(r2))
        r1[i], r2[j] = copy.deepcopy(r2[j]), copy.deepcopy(r1[i])
    return r1, r2

rs_tb.register("mate",   cx_rulesets)
rs_tb.register("mutate", mutate_ruleset)

print("Operators registered for both populations.")

Operators registered for both populations.


## Step 10: Migration — The Coevolution Link

Migration is what makes this *co*evolution rather than two independent GAs.

### Migration Direction 1: Paths → Rule Sets

`seed_rulesets_with_best_paths`  
Take the top N paths from the path pool and inject them into randomly chosen rule sets.  
This ensures the rule set population always has access to the best known building blocks.

### Migration Direction 2: Rule Sets → Paths

`promote_paths_from_rulesets`  
Extract every individual path from the top N rule sets.  
Evaluate them standalone and inject back into the path pool.  

**Why this matters**: A path may score poorly in isolation (low standalone F1)  
but perform excellently *in combination* with other paths.  
This migration surfaces those collaborative paths and lets them compete in the path pool,  
where they may then seed future rule sets.

In [66]:
def seed_rulesets_with_best_paths(path_pop, ruleset_pop, n_best=5):
    """
    Paths → Rule Sets migration.

    Select the top n_best paths from the path population by standalone F1.
    For each selected path, inject a deep copy into a randomly chosen rule set:
      - If the rule set has room (<MAX_PATHS), append the path.
      - Otherwise, replace a random existing path.
    Mark the modified rule set's fitness as invalid so it gets re-evaluated.
    """
    best_paths = tools.selBest(path_pop, n_best)
    for path in best_paths:
        target = random.choice(ruleset_pop)
        if len(target) < MAX_PATHS:
            target.append(copy.deepcopy(path))
        else:
            target[random.randrange(len(target))] = copy.deepcopy(path)
        del target.fitness.values


def promote_paths_from_rulesets(ruleset_pop, path_pop, n_best_rs=3, path_pop_cap=100):
    """
    Rule Sets → Paths migration.

    Select the top n_best_rs rule sets by combined F1.
    Extract every path inside those rule sets, wrap each as a Path individual,
    evaluate standalone F1, and add to the path population.
    Trim the path population to path_pop_cap by keeping the highest-fitness paths.

    Effect: paths that work well in combination — even if weak alone — get a
    chance to compete and potentially seed future rule sets.
    """
    best_rs = tools.selBest(ruleset_pop, n_best_rs)
    for ruleset in best_rs:
        for raw_path in ruleset:
            p = creator.Path(copy.deepcopy(raw_path))
            p.fitness.values = path_tb.evaluate(p)
            path_pop.append(p)
    # Keep only the best paths up to the cap
    path_pop.sort(key=lambda x: x.fitness.values[0], reverse=True)
    del path_pop[path_pop_cap:]

print("Migration functions defined.")

Migration functions defined.


## Step 11: Shared Evolution Step (with Elitism)

Both populations use the same generational loop:
1. **Preserve elite** — copy the top `n_elite` individuals unchanged into the next generation
2. **Select** offspring for the remaining slots via tournament selection
3. **Crossover** pairs with probability `cx_prob`
4. **Mutate** individuals with probability `mut_prob`
5. **Re-evaluate** any individual whose fitness was invalidated
6. Return `elite + offspring` as the new population

Elitism guarantees the best solution found so far is never lost to selection noise
or an unlucky mutation. With `n_elite=1`, only one individual is protected per generation —
enough to prevent regression without reducing selection pressure on the rest.

In [67]:
def evo_step(pop, tb, cx_prob=0.4, mut_prob=0.4, n_elite=1):
    """
    Run one generation of evolution with elitism.

    Parameters
    ----------
    pop      : current population
    tb       : Toolbox (path_tb or rs_tb)
    cx_prob  : crossover probability per pair
    mut_prob : mutation probability per individual
    n_elite  : number of best individuals carried forward unchanged

    Returns
    -------
    next generation population (same size as pop)
    """
    # Carry elite individuals forward unchanged
    elite = [tb.clone(e) for e in tools.selBest(pop, n_elite)]

    # Fill the rest via selection → crossover → mutation
    offspring = list(map(tb.clone, tb.select(pop, len(pop) - n_elite)))

    for c1, c2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < cx_prob:
            tb.mate(c1, c2)
            del c1.fitness.values, c2.fitness.values

    for ind in offspring:
        if random.random() < mut_prob:
            tb.mutate(ind)
            del ind.fitness.values

    for ind in offspring:
        if not ind.fitness.valid:
            ind.fitness.values = tb.evaluate(ind)

    return elite + offspring

print("evo_step with elitism defined.")

evo_step with elitism defined.


## Step 12: Coevolution Loop (with Hall of Fame)

The outer loop alternates between evolving paths and evolving rule sets, with migration
between cycles. Two Hall of Fame objects track the best individuals *ever seen* across
all generations — not just the survivors in the final population.

```
for each coevolution cycle:
    1. Evolve path pool       (inner_path_gens generations, with elitism)
    2. Update path HoF
    3. Migrate: seed rule sets with best standalone paths
    4. Evolve rule set pool   (inner_rs_gens generations, with elitism)
    5. Update rule set HoF
    6. Migrate: promote paths found inside best rule sets back to path pool
```

### Hall of Fame

`tools.HallOfFame(k)` keeps the top k individuals ever evaluated, updated after every
evolution step. The final answer is taken from the HoF rather than the end-of-run
population — this means a great rule set found in cycle 3 that was later displaced by
selection pressure is still returned.

In [68]:
def coevolve(
    path_pop_size    = 60,
    ruleset_pop_size = 40,
    coevo_gens       = 12,
    inner_path_gens  = 5,
    inner_rs_gens    = 5,
    n_migrate        = 5,
    hof_size         = 5,    # top-k individuals tracked in each Hall of Fame
):
    """
    Run cooperative coevolution over two populations.

    Returns
    -------
    best_path    : all-time best path (from path HoF)
    best_ruleset : all-time best rule set (from ruleset HoF)
    path_pop     : final path population
    ruleset_pop  : final rule set population
    path_hof     : Hall of Fame for paths (top hof_size ever seen)
    rs_hof       : Hall of Fame for rule sets (top hof_size ever seen)
    """
    # ── Initialise populations ────────────────────────────────────────────────
    path_pop    = path_tb.population(n=path_pop_size)
    ruleset_pop = rs_tb.population(n=ruleset_pop_size)

    for p in path_pop:    p.fitness.values = path_tb.evaluate(p)
    for r in ruleset_pop: r.fitness.values = rs_tb.evaluate(r)

    # ── Initialise Hall of Fame objects ───────────────────────────────────────
    path_hof = tools.HallOfFame(hof_size)
    rs_hof   = tools.HallOfFame(hof_size)
    path_hof.update(path_pop)
    rs_hof.update(ruleset_pop)

    print(f"Initialized: {path_pop_size} paths, {ruleset_pop_size} rule sets")
    print(f"Running {coevo_gens} coevo cycles "
          f"({inner_path_gens} path gens + {inner_rs_gens} ruleset gens each)\n")

    # ── Coevolution cycles ────────────────────────────────────────────────────
    for gen in range(1, coevo_gens + 1):
        print(f"── Cycle {gen:02d}/{coevo_gens} ──────────────────────────────")

        # 1. Evolve paths
        for _ in range(inner_path_gens):
            path_pop = evo_step(path_pop, path_tb)
        path_hof.update(path_pop)                          # update HoF
        bp = path_hof[0]
        print(f"  [PATH POP]    best standalone F1: {bp.fitness.values[0]:.4f}  "
              f"| {len(bp)} nodes")

        # 2. Migrate: best paths → rule sets
        seed_rulesets_with_best_paths(path_pop, ruleset_pop, n_best=n_migrate)
        for r in ruleset_pop:
            if not r.fitness.valid:
                r.fitness.values = rs_tb.evaluate(r)

        # 3. Evolve rule sets
        for _ in range(inner_rs_gens):
            ruleset_pop = evo_step(ruleset_pop, rs_tb)
        rs_hof.update(ruleset_pop)                         # update HoF
        br = rs_hof[0]
        print(f"  [RULESET POP] best combined score: {br.fitness.values[0]:.4f}  "
              f"| {len(br)} paths, nodes/path: {[len(p) for p in br]}")

        # 4. Migrate: paths from best rule sets → path pool
        promote_paths_from_rulesets(
            ruleset_pop, path_pop, n_best_rs=3, path_pop_cap=path_pop_size
        )

    return path_hof[0], rs_hof[0], path_pop, ruleset_pop, path_hof, rs_hof

## Step 13: Run the Coevolution

In [69]:
best_path, best_ruleset, final_path_pop, final_ruleset_pop, path_hof, rs_hof = coevolve(
    path_pop_size    = 60,
    ruleset_pop_size = 40,
    coevo_gens       = 12,
    inner_path_gens  = 5,
    inner_rs_gens    = 5,
    n_migrate        = 5,
    hof_size         = 5,
)

Initialized: 60 paths, 40 rule sets
Running 12 coevo cycles (5 path gens + 5 ruleset gens each)

── Cycle 01/12 ──────────────────────────────
  [PATH POP]    best standalone F1: 0.7049  | 6 nodes
  [RULESET POP] best combined score: 0.0267  | 2 paths, nodes/path: [1, 3]
── Cycle 02/12 ──────────────────────────────
  [PATH POP]    best standalone F1: 0.7520  | 6 nodes
  [RULESET POP] best combined score: 0.7628  | 2 paths, nodes/path: [7, 2]
── Cycle 03/12 ──────────────────────────────
  [PATH POP]    best standalone F1: 0.7619  | 7 nodes
  [RULESET POP] best combined score: 0.7807  | 2 paths, nodes/path: [5, 9]
── Cycle 04/12 ──────────────────────────────
  [PATH POP]    best standalone F1: 0.7717  | 7 nodes
  [RULESET POP] best combined score: 0.7901  | 2 paths, nodes/path: [8, 7]
── Cycle 05/12 ──────────────────────────────
  [PATH POP]    best standalone F1: 0.7717  | 7 nodes
  [RULESET POP] best combined score: 0.7906  | 2 paths, nodes/path: [7, 7]
── Cycle 06/12 ─────────────

## Step 14: Results

We report two outputs:

1. **Best individual path** — the single strongest standalone decision rule  
2. **Best rule set** — the combination of paths achieving the highest joint F1

The rule set should outperform the single path on recall (more coverage)  
while staying competitive on precision, since each path is individually selective.

In [70]:
# ── Best individual path ───────────────────────────────────────────────────────
print("=" * 70)
print("BEST INDIVIDUAL PATH")
print("=" * 70)
print(pretty_path(best_path))
print(f"Nodes: {len(best_path)}")

path_val_pred  = apply_path(best_path, X_val).astype(int)
path_test_pred = apply_path(best_path, X_test).astype(int)
print(f"\nVal  metrics: {evaluate(y_val,  path_val_pred)}")
print(f"Val  confusion: {confusion(y_val,  path_val_pred)}")
print(f"Test metrics: {evaluate(y_test, path_test_pred)}")
print(f"Test confusion: {confusion(y_test, path_test_pred)}")

# ── Best rule set ──────────────────────────────────────────────────────────────
print()
print("=" * 70)
print("BEST RULE SET")
print("=" * 70)
print(pretty_ruleset(best_ruleset))
print(f"\nTotal paths: {len(best_ruleset)} | Nodes per path: {[len(p) for p in best_ruleset]}")

rs_val_pred  = apply_ruleset(best_ruleset, X_val)
rs_test_pred = apply_ruleset(best_ruleset, X_test)
print(f"\nVal  metrics: {evaluate(y_val,  rs_val_pred)}")
print(f"Val  confusion: {confusion(y_val,  rs_val_pred)}")
print(f"Test metrics: {evaluate(y_test, rs_test_pred)}")
print(f"Test confusion: {confusion(y_test, rs_test_pred)}")

BEST INDIVIDUAL PATH
V10_bin_code <= 8 AND V14 < -0.4273 AND V16 < -0.3403 AND V11 >= 1.6124 AND V12 < -1.9627
Nodes: 5

Val  metrics: {'precision': 0.9122807017543859, 'recall': 0.7323943661971831, 'f1': 0.8125}
Val  confusion: {'TP': 52, 'FP': 5, 'FN': 19, 'TN': 42483}
Test metrics: {'precision': 0.8225806451612904, 'recall': 0.7183098591549296, 'f1': 0.7669172932330827}
Test confusion: {'TP': 51, 'FP': 11, 'FN': 20, 'TN': 42477}

BEST RULE SET
  [01] V9 <= -0.3735 AND V2 < 1.7987 AND V10_bin_code < 5 AND V12 <= -0.7835 AND V3 <= -0.0024 AND V7 <= 0.1240 AND V4 >= -0.1980
OR
  [02] V10_bin_code <= 8 AND V14 < -0.4273 AND V16 < -0.3403 AND V11 >= 1.6124 AND V12 <= -1.9627

Total paths: 2 | Nodes per path: [7, 5]

Val  metrics: {'precision': 0.9166666666666666, 'recall': 0.7746478873239436, 'f1': 0.8396946564885496}
Val  confusion: {'TP': 55, 'FP': 5, 'FN': 16, 'TN': 42483}
Test metrics: {'precision': 0.7910447761194029, 'recall': 0.7464788732394366, 'f1': 0.7681159420289855}
Test conf

## Step 15: Inspect Top Paths from Both Pools

This gives visibility into both populations at the end of evolution:

- **Top paths by standalone F1** — the best building blocks found independently  
- **Top rule sets by combined F1** — the best combinations found collaboratively

Comparing these two lists reveals which paths improved through collaboration:  
paths that rank lower in standalone F1 but appear in top rule sets were  
discovered via the rule set context and promoted through migration.

In [71]:
# ── Hall of Fame: top paths ever seen ─────────────────────────────────────────
print("HALL OF FAME — TOP PATHS (all-time best standalone F1)")
print("-" * 60)
for i, p in enumerate(path_hof, 1):
    print(f"[{i}] F1={p.fitness.values[0]:.4f} | {len(p)} nodes")
    print(f"    {pretty_path(p)}")
    print()

# ── Hall of Fame: top rule sets ever seen ─────────────────────────────────────
print("HALL OF FAME — TOP RULE SETS (all-time best combined score)")
print("-" * 60)
for i, rs in enumerate(rs_hof, 1):
    # Report true F1 separately from the composite score stored in fitness
    rs_pred = apply_ruleset(rs, X_val)
    true_f1 = f1_score(y_val, rs_pred, zero_division=0)
    print(f"[{i}] score={rs.fitness.values[0]:.4f} | true val F1={true_f1:.4f} "
          f"| {len(rs)} paths")
    for j, p in enumerate(rs, 1):
        print(f"     path {j}: {pretty_path(p)}")
    print()

HALL OF FAME — TOP PATHS (all-time best standalone F1)
------------------------------------------------------------
[1] F1=0.8125 | 5 nodes
    V10_bin_code <= 8 AND V14 < -0.4273 AND V16 < -0.3403 AND V11 >= 1.6124 AND V12 < -1.9627

[2] F1=0.8125 | 5 nodes
    V10_bin_code <= 8 AND V14 <= -0.4273 AND V16 < -0.3403 AND V11 >= 1.6124 AND V12 <= -1.9627

[3] F1=0.8125 | 5 nodes
    V10_bin_code <= 8 AND V14 < -0.4273 AND V16 <= -0.3403 AND V11 >= 1.6124 AND V12 <= -1.9627

[4] F1=0.8125 | 5 nodes
    V10_bin_code <= 8 AND V14 < -0.4273 AND V16 < -0.1218 AND V11 >= 1.6124 AND V12 <= -1.9627

[5] F1=0.8125 | 5 nodes
    V10_bin_code <= 8 AND V14 < -0.4273 AND V16 < -0.3403 AND V11 >= 1.6124 AND V12 <= -1.9627

HALL OF FAME — TOP RULE SETS (all-time best combined score)
------------------------------------------------------------
[1] score=0.8476 | true val F1=0.8397 | 2 paths
     path 1: V9 <= -0.3735 AND V2 < 1.7987 AND V10_bin_code < 5 AND V12 <= -0.7835 AND V3 <= -0.0024 AND V7 <= 0.1

## Step 16: Export Results

Save the best path and best rule set to JSON for downstream use or reporting.

In [72]:
def path_to_dict(path):
    return [
        {
            "feature":   col,
            "op":        op,
            "threshold": int(thr) if col in CATEGORICAL_FEATURES else thr,
        }
        for col, op, thr in path
    ]

results = {
    "top_features": TOP_FEATURES,
    "categorical_features": list(CATEGORICAL_FEATURES),
    "best_path": {
        "conditions":   path_to_dict(best_path),
        "n_nodes":      len(best_path),
        "val_metrics":  evaluate(y_val,  path_val_pred),
        "test_metrics": evaluate(y_test, path_test_pred),
    },
    "best_ruleset": {
        "paths":          [path_to_dict(p) for p in best_ruleset],
        "n_paths":        len(best_ruleset),
        "nodes_per_path": [len(p) for p in best_ruleset],
        "val_metrics":    evaluate(y_val,  rs_val_pred),
        "test_metrics":   evaluate(y_test, rs_test_pred),
    },
}

OUTPUT_FILE = "coevo_ga_results.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved to {os.path.abspath(OUTPUT_FILE)}")

Saved to c:\Users\schyu\Documents\Personal Files\UIC\IDS 560\Sandbox\coevo_ga_results.json
